# Table of Contents
- [Import Libraries](#import-libraries)
- [Loading The Cleaned Data](#loading-the-data)
- [Chart 1 - Hourly time-lapse of user movement](#chart-1)
- [Chart 2 - Coverage map per operator](#chart-2)
- [Chart 3 - Downlink traffic hotspots per operator](#chart-3)
- [Chart 4 - RSRP per device manufacturer per operator](#chart-4)
- [Findings and key insights](#findings)
  - [1. Coverage: C leads, A trails, and A carries the most users](#finding-1)
  - [2. Traffic: B moves the data, A moves the people](#finding-2)
  - [3. Downlink is extremely concentrated, and the hotspots are B's](#finding-3)
  - [4. The city breathes: a 6x day/night swing](#finding-4)
  - [5. Handsets: the ranking is real, the tail is not](#finding-5)

<a id="import-libraries"></a>
### Import Libraries

In [18]:
import math

import pandas as pd
import plotly.express as px
from ipywidgets import interact, Dropdown, IntSlider, SelectionSlider

<a id="loading-the-data"></a>
### Loading The Cleaned Data

In [19]:
rsrp = pd.read_parquet('../rsrp_clean.parquet',
                       columns=['Timestamp', 'LocationLatitude', 'LocationLongitude',
                                'RadioOperatorName', 'RSRP'])

rsrp['Hour'] = rsrp['Timestamp'].dt.hour
rsrp.shape

(1975645, 6)

Riyadh map centre, reused by every map in this notebook.

In [20]:
RIYADH = {'lat': 24.6967, 'lon': 46.7119}

<a id="chart-1"></a>
### Chart 1 - Hourly time-lapse of user movement
A density map animated over the local hour of day, built from the RSRP sample
locations.

The 2M raw samples cannot be sent to the browser 24 times over, so the points are
first snapped to a grid by rounding the coordinates to 3 decimals (~110 m) and
counted per hour. That is ~5k points per frame instead of 2M.

In [21]:
grid = (rsrp
        .assign(lat=rsrp['LocationLatitude'].round(3),
                lon=rsrp['LocationLongitude'].round(3))
        .groupby(['Hour', 'lat', 'lon'])
        .size()
        .reset_index(name='Samples')
        .sort_values('Hour'))

grid.shape

(122963, 4)

`sort_values('Hour')` is not cosmetic: plotly builds the animation frames in the
order it meets the rows, so unsorted data gives frames in a random order.

`range_color` is capped at the 99th percentile because a handful of very hot cells
would otherwise take the whole colour scale and leave every frame looking empty.

In [ ]:
px.density_map(grid, lat='lat', lon='lon', z='Samples',
               animation_frame='Hour',
               radius=6, zoom=9.5, center=RIYADH,
               range_color=[0, grid['Samples'].quantile(0.99)],
               map_style='carto-positron', height=650,
               title='User density by hour of day (Riyadh, local time)')

<a id="chart-2"></a>
### Chart 2 - Coverage map per operator
Sample locations for one selected operator, coloured by the RSRP measured there.

Same 3-decimal grid as chart 1, but the cells now carry the **mean RSRP** instead
of a sample count. Cells with fewer than 3 samples are dropped - a cell holding a
single reading is noise, and those cells are what make a coverage map look
randomly speckled.

In [23]:
cover = (rsrp
         .assign(lat=rsrp['LocationLatitude'].round(3),
                 lon=rsrp['LocationLongitude'].round(3))
         .groupby(['RadioOperatorName', 'lat', 'lon'])['RSRP']
         .agg(['mean', 'size'])
         .reset_index())

cover = cover[cover['size'] >= 3]
cover.groupby('RadioOperatorName').size()

RadioOperatorName
Operator A    23392
Operator B    14456
Operator C    13303
dtype: int64

The colour range is **hard-coded, not auto-scaled**. If plotly picks the range per
operator, every operator ends up looking equally green and the dropdown compares
nothing. `[-105, -70]` is roughly the 5th-95th percentile across all three, so the
colours mean the same dBm no matter which operator is selected.

This is written as a function because it becomes the Dash callback body later.

In [ ]:
def coverage_map(operator):
    d = cover[cover['RadioOperatorName'] == operator]
    return px.scatter_map(d, lat='lat', lon='lon', color='mean',
                          color_continuous_scale='RdYlGn',
                          range_color=[-105, -70],
                          hover_data={'size': True},
                          zoom=9.5, center=RIYADH, opacity=0.7,
                          map_style='carto-positron', height=650,
                          labels={'mean': 'RSRP (dBm)', 'size': 'Samples'},
                          title=f'Mean RSRP by location - {operator}')


coverage_map('Operator A')

The task asks for a **dropdown** to pick the operator, so `coverage_map` is
driven by an `ipywidgets` dropdown rather than by editing the argument by hand.
Operator C's map should be visibly greener.

(The widget needs a running kernel - open the notebook and run it. A static
render of the `.ipynb` shows the last drawn figure only.)

In [ ]:
operators = sorted(cover['RadioOperatorName'].unique())


@interact(operator=Dropdown(options=operators, value=operators[0],
                            description='Operator:'))
def show_coverage(operator):
    coverage_map(operator).show()

<a id="chart-3"></a>
### Chart 3 - Downlink traffic hotspots per operator
Riyadh is cut into H3 hexagons, the downlink traffic falling in each hexagon is
summed per operator, and each hexagon is drawn as a bubble at its centre sized by
that total. The slider changes the H3 resolution, i.e. the area each bubble covers.

In [ ]:
import h3

In [ ]:
traffic = pd.read_parquet('../traffic_clean.parquet',
                          columns=['LocationLatitude', 'LocationLongitude',
                                   'RadioOperatorName', 'TrafficDirection',
                                   'TrafficVolume'])

downlink = traffic[traffic['TrafficDirection'] == 'Downlink']
downlink.shape

(64945, 5)

Each resolution is a different hexagon area, so all four are built once up front.
Re-binning inside the callback would make the slider crawl.

In [ ]:
for res in range(6, 10):
    print(res, round(h3.average_hexagon_area(res, unit='km^2'), 3), 'km2')

6 36.129 km2
7 5.161 km2
8 0.737 km2
9 0.105 km2


In [ ]:
hexes = {}

for res in range(6, 10):
    cell = [h3.latlng_to_cell(la, lo, res)
            for la, lo in zip(downlink['LocationLatitude'], downlink['LocationLongitude'])]

    g = (downlink.assign(Hex=cell)
                 .groupby(['RadioOperatorName', 'Hex'])['TrafficVolume']
                 .sum()
                 .reset_index())

    g[['lat', 'lon']] = pd.DataFrame([h3.cell_to_latlng(c) for c in g['Hex']],
                                     index=g.index)
    hexes[res] = g

{res: len(g) for res, g in hexes.items()}

{6: 272, 7: 1112, 8: 4139, 9: 9913}

`size_max` is what keeps the bubbles readable: plotly scales the largest value in
the frame to that many pixels. Without it a single 100 GB hexagon covers the city.

`TrafficVolume` is in MB, so it is converted to GB **as a named column** - passing
a bare `series / 1024` to `size=` leaves the hover and legend labelled
`TrafficVolume`, because the label lookup keys off the column name.

All three operators share the same hexagon centre, so the largest bubble would sit
on top of the other two and hide exactly the comparison the chart exists to make.
Each operator is therefore nudged a fraction of a hexagon radius off the centre,
at its own angle. The offset is cosmetic - the binning itself is untouched.

In [ ]:
def traffic_bubbles(res):
    g = hexes[res].copy()
    g['Downlink (GB)'] = g['TrafficVolume'] / 1024

    # nudge each operator off the shared hexagon centre so bubbles do not occlude
    ops = sorted(g['RadioOperatorName'].unique())
    radius_km = math.sqrt(h3.average_hexagon_area(res, unit='km^2') / math.pi)
    offset = 0.35 * radius_km / 111.0                      # km -> degrees latitude
    angle = {o: 2 * math.pi * i / len(ops) for i, o in enumerate(ops)}
    a = g['RadioOperatorName'].map(angle)

    g['plot_lat'] = g['lat'] + offset * a.map(math.sin)
    g['plot_lon'] = g['lon'] + offset * a.map(math.cos) / math.cos(math.radians(RIYADH['lat']))

    return px.scatter_map(g, lat='plot_lat', lon='plot_lon',
                          size='Downlink (GB)',
                          color='RadioOperatorName',
                          size_max=45, zoom=9.5, center=RIYADH, opacity=0.6,
                          map_style='carto-positron', height=650,
                          hover_name='Hex',
                          hover_data={'plot_lat': False, 'plot_lon': False,
                                      'Downlink (GB)': ':.1f'},
                          labels={'RadioOperatorName': 'Operator'},
                          title=f'Downlink traffic per operator - H3 resolution {res} '
                                f'({h3.average_hexagon_area(res, unit="km^2"):.2f} km2 per hexagon)')


traffic_bubbles(7)

The task asks for a **slider** over the area each bubble covers. Resolution 6 is
~36 km2 per hexagon and shows who owns the city; resolution 9 is ~0.1 km2 and
shows individual hotspots.

In [ ]:
@interact(res=IntSlider(min=6, max=9, step=1, value=7, continuous_update=False,
                       description='H3 res:'))
def show_bubbles(res):
    traffic_bubbles(res).show()

<a id="chart-4"></a>
### Chart 4 - RSRP per device manufacturer per operator
Grouped bars, one group per manufacturer and one bar per operator. The dropdown
picks the aggregation and the slider sets how many samples a manufacturer must
have on that operator before it is shown.

The data has no user identifier, so *number of users* is approximated by the
**number of samples**. This has to be stated in the findings - a single commuter
logging all day looks like many users.

In [ ]:
dev = pd.read_parquet('../rsrp_clean.parquet',
                      columns=['RadioOperatorName', 'DeviceManufacturer', 'RSRP'])

# the manufacturer was lower-cased during cleaning to merge Lenovo/LENOVO/lenovo;
# title-case it back for the axis labels
dev['DeviceManufacturer'] = dev['DeviceManufacturer'].str.title()

stats = (dev.groupby(['DeviceManufacturer', 'RadioOperatorName'])['RSRP']
            .agg(Average='mean',
                 Minimum='min',
                 Maximum='max',
                 **{'90th percentile': lambda s: s.quantile(0.9)},
                 Samples='size')
            .round(1)
            .reset_index())

stats.head()

All four aggregations are computed once, so the dropdown only selects a column.

Two details that decide whether this chart is readable:
- plotly draws bars from **zero**, and RSRP is negative, so without an explicit
  y-axis range you get 24 near-identical bars hanging off the top of the plot.
  The range is derived from the selected metric, **not hard-coded**. A fixed
  `[-125, -50]` window breaks two of the four required aggregations: `Minimum`
  bottoms out at the -140 dBm cleaning floor, so 16 of its bars fall off the
  bottom of the plot, and `Maximum` reaches -44 dBm, so 7 of its bars run off
  the top.
- the manufacturers are sorted by the selected metric, otherwise the ranking the
  chart exists to show is buried in alphabetical order.

In [ ]:
def device_bars(metric, min_samples):
    d = stats[stats['Samples'] >= min_samples]
    if d.empty:
        raise ValueError(f'no manufacturer has {min_samples:,} samples on any operator')

    order = d.groupby('DeviceManufacturer')[metric].mean().sort_values(ascending=False).index

    fig = px.bar(d, x='DeviceManufacturer', y=metric, color='RadioOperatorName',
                 barmode='group', height=550,
                 category_orders={'DeviceManufacturer': list(order)},
                 hover_data={'Samples': ':,'},
                 labels={'DeviceManufacturer': 'Manufacturer',
                         'RadioOperatorName': 'Operator',
                         metric: f'RSRP - {metric} (dBm)'},
                 title=f'RSRP {metric} by manufacturer '
                       f'(at least {min_samples:,} samples per operator)')

    # fit the window to the metric actually being shown
    lo, hi = d[metric].min(), d[metric].max()
    pad = max(2.0, 0.05 * (hi - lo))
    fig.update_yaxes(range=[lo - pad, hi + pad])
    return fig


device_bars('Average', 1000)

The dropdown picks the aggregation and the slider sets the minimum sample count.
A low threshold brings back the long tail of rare handsets, and their bars swing
wildly - that swing is sample size, not coverage.

In [ ]:
THRESHOLDS = [10, 50, 100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]


@interact(metric=Dropdown(options=['Average', 'Minimum', 'Maximum', '90th percentile'],
                          value='Average', description='Aggregate:'),
          min_samples=SelectionSlider(options=THRESHOLDS, value=1_000,
                                      continuous_update=False,
                                      description='Min samples:'))
def show_devices(metric, min_samples):
    device_bars(metric, min_samples).show()

<a id="findings"></a>
# Findings and key insights

Every number quoted below is produced by the cells in this section, from the same
cleaned Parquet files the four charts use: **1.98 M** valid 4G RSRP samples and
**129.8 k** traffic samples, all in Riyadh, spanning
**2019-11-01 21:15 to 2019-11-04 23:59 local time** - a little over three days,
not the full week the brief mentions.

<a id="finding-1"></a>
## 1. Coverage: C leads, A trails, and A carries the most users

The ranking is consistent across every way of cutting it - mean, median, the
bottom decile, and the share of samples in the poor bands.

In [ ]:
coverage_summary = (rsrp.groupby('RadioOperatorName')['RSRP']
                        .agg(Samples='size', Mean='mean', Median='median',
                             P10=lambda s: s.quantile(0.10))
                        .round(1))
coverage_summary['Share of samples %'] = (coverage_summary['Samples']
                                          / coverage_summary['Samples'].sum() * 100).round(1)
coverage_summary['Below -100 dBm %'] = (rsrp.assign(bad=rsrp['RSRP'] < -100)
                                            .groupby('RadioOperatorName')['bad']
                                            .mean() * 100).round(1)
coverage_summary

,Samples,Mean,Median,P10,Share of samples %,Below -100 dBm %
RadioOperatorName,,,,,,
Operator A,1079060,-85.5,-83.0,-107.0,54.6,17.1
Operator B,445993,-84.6,-85.0,-100.0,22.6,9.1
Operator C,450592,-80.9,-80.0,-96.0,22.8,4.5


**Operator C has the best coverage by a clear margin.** Its mean RSRP is
**-80.9 dBm** against **-84.6** for B and **-85.5** for A, and only **4.5 %** of
its samples fall below -100 dBm - against **9.1 %** for B and **17.1 %** for A.
The bottom decile is the sharpest separator: C's 10th percentile is **-96 dBm**,
A's is **-107 dBm**. C's worst 10 % of locations are better than A's worst 10 %
by roughly **11 dB**, which is more than a tenfold difference in received power.

**The operator with the weakest coverage serves the most users.** A accounts for
**54.6 %** of all RSRP samples - more than B and C combined - while posting the
worst coverage of the three. Whether that is load-driven degradation or simply a
wider, thinner footprint cannot be settled from this data, but it is the single
most commercially interesting line in the dashboard: A's coverage problem affects
the largest population.

On the **Chart 2** dropdown this is visible directly - switch from A to C and the
map turns from mottled orange to broadly green, with A's red patches clustered on
the city outskirts rather than in the centre.

<a id="finding-2"></a>
## 2. Traffic: B moves the data, A moves the people

Sample share and traffic share point in opposite directions.

In [ ]:
traffic_summary = (traffic.pivot_table(index='RadioOperatorName',
                                       columns='TrafficDirection',
                                       values='TrafficVolume',
                                       aggfunc='sum') / 1024).round(1)
traffic_summary['DL / UL ratio'] = (traffic_summary['Downlink']
                                    / traffic_summary['Uplink']).round(1)
traffic_summary['Share of downlink %'] = (traffic_summary['Downlink']
                                          / traffic_summary['Downlink'].sum() * 100).round(1)
traffic_summary['Share of samples %'] = (traffic['RadioOperatorName']
                                         .value_counts(normalize=True) * 100).round(1)
traffic_summary

TrafficDirection,Downlink,Uplink,DL / UL ratio,Share of downlink %,Share of samples %
RadioOperatorName,,,,,
Operator A,691.1,63.4,10.9,23.9,49.6
Operator B,1869.9,104.9,17.8,64.6,28.7
Operator C,334.0,40.4,8.3,11.5,21.7


**Operator B carries 64.6 % of all downlink traffic from 28.7 % of the samples.**
Operator A is the mirror image: **49.6 %** of the samples but only **23.9 %** of
the downlink gigabytes. Per sample, a B user pulls roughly **six times** the
downlink of an A user.

The **DL/UL ratio** says the same thing about usage mix. B sits at **17.8:1**,
A at **10.9:1**, C at **8.3:1**. A ratio that high is the signature of heavy video
and download consumption; C's lower ratio points at a more conversational,
upload-balanced traffic profile.

Read together with finding 1, the two operators are solving different problems.
**B is a capacity story** - the most data on the fewest samples, so its risk is
congestion. **A is a coverage story** - the most users on the weakest signal, so
its risk is that those users cannot use the data they are paying for.

<a id="finding-3"></a>
## 3. Downlink is extremely concentrated, and the hotspots are B's

At H3 resolution 7 (~5 km2 per hexagon) Riyadh's downlink traffic lands in 470
hexagons - but it is not spread across them.

In [ ]:
hex7 = (hexes[7].groupby('Hex')['TrafficVolume'].sum()
                            .sort_values(ascending=False))
print(f'hexagons with downlink traffic : {len(hex7)}')
print(f'top 10 share of all downlink   : {hex7.head(10).sum() / hex7.sum():.1%}')

top5 = (hexes[7][hexes[7]['Hex'].isin(hex7.head(5).index)]
        .pivot_table(index='Hex', columns='RadioOperatorName',
                     values='TrafficVolume', aggfunc='sum', fill_value=0)
        .reindex(hex7.head(5).index) / 1024).round(1)
top5.index.name = 'Hex (GB downlink)'
top5

**Ten hexagons out of 470 carry 60.5 % of Riyadh's downlink traffic.** Roughly
50 km2 of a city of thousands accounts for two thirds of the data. For a capacity
planner this is the whole ballgame: investment targeted at ten cells buys more
than anything spread evenly across the map.

**Those hotspots are almost entirely single-operator.** The largest hexagon
carries **805 GB for Operator B** against **0.9 GB for A** and **0.1 GB for C** -
a ratio of nearly 900:1. That is far too lopsided to be a genuine difference in
demand at one location; it is much more likely a single very heavy user or a
small cluster of them on B, which is a caution about reading these bubbles as
market share. The second and third hotspots are more mixed (B ~200 GB with C at
~42 GB, and B ~103 GB with C ~30 GB), so the effect is not uniform.

Drag the **Chart 3** slider from resolution 6 to 9 to see this resolve: at res 6
the map shows three or four broad blobs over the city, and by res 9 the traffic
collapses onto a handful of pinpoints.

<a id="finding-4"></a>
## 4. The city breathes: a 6x day/night swing

In [ ]:
hourly = rsrp.groupby('Hour').agg(Samples=('RSRP', 'size'),
                                  Median_RSRP=('RSRP', 'median')).round(1)
print(f"peak   : {hourly['Samples'].idxmax():02d}:00  {hourly['Samples'].max():,} samples")
print(f"trough : {hourly['Samples'].idxmin():02d}:00  {hourly['Samples'].min():,} samples")
print(f"ratio  : {hourly['Samples'].max() / hourly['Samples'].min():.1f}x")
hourly.T

peak   : 16:00  140,070 samples
trough : 04:00  22,503 samples
ratio  : 6.2x


Hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
Samples,39212.0,44467.0,31489.0,22826.0,22503.0,30739.0,52336.0,89053.0,95650.0,84961.0,...,89677.0,98859.0,140070.0,133188.0,137899.0,110591.0,104540.0,104147.0,103580.0,68524.0
Median_RSRP,-83.0,-80.0,-83.0,-79.0,-78.0,-77.0,-83.0,-83.0,-83.0,-83.0,...,-83.0,-85.0,-84.0,-84.0,-83.0,-83.0,-81.0,-82.0,-82.0,-87.0


Activity bottoms out at **04:00 (22.5 k samples)** and peaks at
**16:00 (140.1 k samples)** - a **6.2x** swing. The shape is distinctly local:
a slow start, a broad plateau through the working day, and the true maximum in the
**late afternoon and evening block, 16:00-18:00**, which stays elevated past
22:00. This is a Riyadh evening-social pattern, not the twin commuter spikes a
European city would show.

Two things follow for the network. First, the busy hour to plan against is
**16:00-18:00**, not the morning. Second, the **Chart 1** time-lapse shows *where*
that mass sits: the overnight frames concentrate in the residential ring, and
through the day density migrates inward toward the commercial core before
spreading back out in the evening.

Median RSRP moves in the opposite direction to the crowd - **-77 to -78 dBm** in
the 04:00-05:00 dead hours against **-84 to -85 dBm** across the busy afternoon.
That 6-7 dB gap is worth flagging but **not worth over-reading**: the quiet hours
have a fraction of the samples and are dominated by stationary indoor devices
sitting on a nearby cell, so the comparison mixes network load with a change in
who is measuring and where.

<a id="finding-5"></a>
## 5. Handsets: the ranking is real, the tail is not

In [ ]:
share = (stats.groupby('DeviceManufacturer')['Samples'].sum()
              .sort_values(ascending=False))
print('share of all samples:')
print((share / share.sum()).head(5).round(3))

solid = stats[stats['Samples'] >= 1_000]
rank = solid.groupby('DeviceManufacturer')['Average'].mean().sort_values().round(1)
print(f'\nmanufacturers with >=1,000 samples on an operator: {len(rank)}')
print('\nweakest:'); print(rank.head(5))
print('\nstrongest:'); print(rank.tail(5))

**Samsung is 89.1 % of every sample in the file.** Huawei is a distant second at
**3.0 %**, and nothing else clears 1.5 %. This single fact should govern how the
rest of Chart 4 is read, and it is exactly why the chart has a minimum-samples
slider.

Among the 13 manufacturers with at least 1,000 samples on an operator, the spread
of mean RSRP is about **14 dB**:

- **weakest** - HMD Global (**-89.9**), Sony (**-89.0**), Lenovo (**-88.5**),
  Motorola (**-88.2**), OnePlus (**-87.2**)
- **strongest** - TCL (**-75.9**), Huawei (**-80.2**), Xiaomi (**-83.2**),
  Samsung (**-83.6**), Oppo (**-83.7**)

The **HMD Global / Sony / Lenovo cluster sits 4-6 dB below Samsung**, which is a
real and actionable gap: those handsets consistently report worse coverage on the
same network, in the same city, over the same days. For a care team, complaints
from those models are more likely to be handset-related than cell-related.

Two warnings, both demonstrable with the slider:

1. **TCL's -75.9 dBm is not a finding.** Drag the minimum-samples slider up and it
   disappears. Brands with a few thousand samples are measured by a handful of
   devices in a handful of places; their bar is a location average wearing a
   manufacturer's name.
2. **Samsung is the baseline, not a competitor.** At 89 % of the data, Samsung's
   average *is* the network average (-84.2 dBm for Samsung against -84.5 dBm for
   everything else combined). Any brand comparison is really a comparison against
   Samsung, and the honest reading of the chart is "which brands deviate from the
   network", not "which brand is best".